# N6 v2 - Margin-based reject option

**Question.** Can a snapshot classifier recognise the faults it is likely to get wrong, and
abstain on them? Two scores rank the test snapshots:

* `prob`   - the largest class probability of LightGBM (classifier confidence);
* `margin` - a margin in noise units: features are whitened by the measurement-noise standard
  deviation, and the margin is the distance to the nearest training snapshots of the second
  closest fault type minus the distance to those of the closest type. It does not use the
  classifier, so it can be computed for any model.

For each score: risk-coverage curve, area under it (AURC, lower is better), the LARGEST
coverage whose retained accuracy meets 95 / 99 / 99.5 %, and the share of high-impedance
faults (R_f > 10 ohm) among the rejected snapshots.

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from lightgbm import LGBMClassifier

K_NN = 5            # neighbours per class for the margin
COV_GRID = np.round(np.linspace(0.05, 1.0, 20), 3)
TARGETS = (0.95, 0.99, 0.995)
REJECT_AT = 0.90    # coverage at which the share of HIF among rejected snapshots is reported


def noise_sd(ds, obs, noise, rep="FPS"):
    """Per-feature standard deviation of the measurement noise, from two independent draws."""
    X1 = K.features_of(ds, obs, 0, noise, rep)
    X2 = K.features_of(ds, obs, 1, noise, rep)
    sd = np.nanstd(X1 - X2, axis=0) / np.sqrt(2.0)
    return np.where(sd > 1e-12, sd, 1e-12)


def margin_scores(Xtr, ytr, Xte, classes):
    """Distance to the K_NN nearest training snapshots of each class (mean), whitened input.
    Returns (pred_by_margin, margin) for the test snapshots."""
    D = np.full((len(Xte), len(classes)), np.inf)
    for j, c in enumerate(classes):
        Xc = Xtr[ytr == c]
        if len(Xc) == 0:
            continue
        nn = NearestNeighbors(n_neighbors=min(K_NN, len(Xc))).fit(Xc)
        d, _ = nn.kneighbors(Xte)
        D[:, j] = d.mean(1)
    order = np.argsort(D, axis=1)
    first = D[np.arange(len(D)), order[:, 0]]
    second = D[np.arange(len(D)), order[:, 1]]
    return np.asarray(classes)[order[:, 0]], second - first


def risk_coverage(score, correct):
    """Accuracy of the retained fraction, keeping the highest scores first."""
    o = np.argsort(-score, kind="stable")
    c = correct[o].astype(float)
    cum = np.cumsum(c) / np.arange(1, len(c) + 1)
    return o, cum


def summarise(score, correct, rf):
    o, cum = risk_coverage(score, correct)
    n = len(o)
    out = {"aurc": float(np.mean(1 - cum))}
    for cov in COV_GRID:
        out[f"acc@{cov:.2f}"] = float(cum[max(1, int(round(cov * n))) - 1])
    for t in TARGETS:
        ok = np.where(cum >= t)[0]
        out[f"cov@{t}"] = float((ok[-1] + 1) / n) if len(ok) else 0.0   # LARGEST coverage
    n_keep = int(round(REJECT_AT * n))
    rej = o[n_keep:]
    out["hif_share_rejected"] = float(np.mean(rf[rej] > 10)) if len(rej) else float("nan")
    out["hif_share_all"] = float(np.mean(rf > 10))
    out["err_share_in_rejected"] = float((~correct[rej]).sum() / max(1, (~correct).sum()))
    return out


In [ ]:
rows = []
feeders = K.feeder_list(SMOKE)
for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    gi = K.rows_for_buses(ds, buses)
    _, marg, _ = P.margins(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0],
                           ang_sigma_deg=K.DESIGN_NOISE[1])
    for k in K.k_list(nb, SMOKE):
        obs = [gi[i] for i in P.dmp(marg, k)]
        for noise in K.NOISE_MAIN:
            key = {"feeder": nb, "k": k, "noise": noise}
            if K.done("n6v2", key):
                rows += K.load_unit("n6v2", key)["rows"]
                continue
            t = time.time()
            X = K.features_of(ds, obs, 0, noise, "FPS")
            X = np.nan_to_num(X)
            Xw = X / noise_sd(ds, obs, noise)
            y, rf = ds.y, ds.rf
            classes = np.unique(y)
            pred = np.empty_like(y); prob = np.zeros(len(y))
            mpred = np.empty_like(y); marg_s = np.zeros(len(y))
            skf = StratifiedKFold(5, shuffle=True, random_state=0)
            for tr, te in skf.split(X, y):
                m = LGBMClassifier(**K.FAST).fit(X[tr], y[tr])
                pr = m.predict_proba(X[te])
                pred[te] = m.classes_[pr.argmax(1)]
                prob[te] = pr.max(1)
                mp, ms = margin_scores(Xw[tr], y[tr], Xw[te], classes)
                mpred[te], marg_s[te] = mp, ms
            correct = pred == y
            unit = []
            for score_name, score in (("prob", prob), ("margin", marg_s)):
                r = dict(feeder=nb, k=k, noise=noise, score=score_name,
                         acc_full=float(correct.mean()),
                         acc_margin_classifier=float((mpred == y).mean()))
                r.update(summarise(score, correct, rf))
                unit.append(r)
            K.save_unit("n6v2", key, {"rows": unit, "meta": K.meta()})
            rows += unit
            u = {x["score"]: x for x in unit}
            print(f"IEEE-{nb} k={k} {noise:7s} acc {correct.mean()*100:5.1f}% | "
                  f"AURC prob {u['prob']['aurc']:.4f} margin {u['margin']['aurc']:.4f} | "
                  f"cov@99% prob {u['prob']['cov@0.99']*100:5.1f}% margin {u['margin']['cov@0.99']*100:5.1f}% | "
                  f"HIF share of rejected (90% cov) prob {u['prob']['hif_share_rejected']*100:4.1f}% "
                  f"margin {u['margin']['hif_share_rejected']*100:4.1f}% (all {u['prob']['hif_share_all']*100:4.1f}%) "
                  f"[{time.time()-t:.0f}s]")


## Accuracy under assumed field distributions of fault resistance

No published field distribution of R_f for distribution feeders was found in the reference
set, so the weights below are ASSUMPTIONS, reported side by side with the uniform simulation
prior. They must be described as assumptions in the paper.

In [ ]:
PRIORS = {
    "uniform (simulation)": None,
    "assumed A: 0-1/1-10/10-100 ohm = 0.55/0.30/0.15": [((0, 1), .55), ((1, 10), .30), ((10, 1e9), .15)],
    "assumed B: 0-1/1-10/10-100 ohm = 0.30/0.40/0.30": [((0, 1), .30), ((1, 10), .40), ((10, 1e9), .30)],
}
fw_rows = []
for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    gi = K.rows_for_buses(ds, buses)
    _, marg, _ = P.margins(cols, Vp, Vf, meta, mag_sigma=K.DESIGN_NOISE[0],
                           ang_sigma_deg=K.DESIGN_NOISE[1])
    obs = [gi[i] for i in P.dmp(marg, 3)]
    for noise in K.NOISE_MAIN:
        X = np.nan_to_num(K.features_of(ds, obs, 0, noise, "FPS"))
        r = K.folds_acc(X, ds.y, ds.rf, seeds=(0,), return_oof=True)
        correct = (r["oof"] == ds.y).astype(float)
        for name, bands in PRIORS.items():
            w = np.ones(ds.n)
            if bands:
                for (lo, hi), tgt in bands:
                    m = (ds.rf >= lo) & (ds.rf < hi)
                    if m.any():
                        w[m] = tgt / m.mean()
            fw_rows.append(dict(feeder=nb, k=3, noise=noise, prior=name,
                                acc=float((correct * w).sum() / w.sum())))
            print(f"IEEE-{nb} {noise:7s} {name:50s} {(correct*w).sum()/w.sum()*100:5.1f}%")


In [ ]:
import pandas as pd
pd.DataFrame(rows).to_csv(os.path.join(WORK, "N6v2_reject.csv"), index=False)
pd.DataFrame(fw_rows).to_csv(os.path.join(WORK, "N6v2_field_weighted.csv"), index=False)
print("wrote N6v2_reject.csv, N6v2_field_weighted.csv")
print("TOTAL", round((time.time() - t_start) / 60, 1), "min")
